# Certified Fair Summarization — a guided tour

**What this notebook is.** A walkthrough of the `fairsum` package: what each piece does,
and — more importantly — *why it is shaped that way*. Almost every design decision here is
a direct response to something that went wrong in the v1 notebooks
(`drug_summary_01_Method_A.ipynb`, `04_2pilot_experiment_fairsum_metrics (1).ipynb`).

**The research question.** Eight patients take the same drug. Three say it worked wonderfully,
two say it barely helped, three say it made them sick. A model writes one paragraph summarising
all eight. Does that paragraph preserve the proportions of the room — or does it quietly drop
the one patient who had a severe reaction?

**Why v2 exists.** v1 asked whether prompt engineering makes summaries fairer, and could not
answer, for two reasons: the measurement instrument was the generator grading its own work, and
the metrics were saturated. This notebook shows how each of those was diagnosed and fixed.

**A warning about the numbers.** Nothing in this notebook is a research result. The generation
cells run on a small slice, and the adverse events are keyword placeholders, not the gold
annotations that Week 2–3 will produce. This is plumbing, explained.

---

### How to read this

| Section | What you will learn |
|---|---|
| 1 | Setup |
| 2 | **The bug that put v1 on hold** — a silent failure scored as a result |
| 3 | The fairness metrics, and why three of them are one metric |
| 4 | Loading the review pool |
| 5 | Building instances, and a design tradeoff measured on real data |
| 6 | Prompts, and why 42% of reasoning outputs are unusable at 1B |
| 7 | Batched generation (needs a GPU — skippable) |
| 8 | The verifier: checking whether a summary conveys an adverse event |
| 9 | Conformal risk control: the guarantee, and what it does *not* say |
| 10 | H1 — what the actual contribution is now |

## 1. Setup

The logic lives in the `fairsum` package (`src/fairsum/`), not in this notebook. That is
deliberate: v1 defined its metrics inline in two notebooks, they drifted apart, and neither
version had a single test. `fairsum` has 228.

Install once from the project root:

```bash
pip install -e ".[dev]"      # pure logic + tests
pip install -e ".[dev,gen]"  # adds torch/transformers for section 7
```

In [ ]:
import numpy as np
import pandas as pd

pd.set_option("display.width", 110)
pd.set_option("display.max_columns", 20)

import fairsum
print("fairsum", fairsum.__version__)

# Section 7 needs a GPU; everything else runs on CPU in seconds.
try:
    import torch
    HAS_GPU = torch.cuda.is_available()
    print("GPU:", torch.cuda.get_device_name(0) if HAS_GPU else "none")
except ImportError:
    HAS_GPU = False
    print("torch not installed - section 7 will be skipped")

## 2. The bug that put v1 on hold

This is the most important section. Everything else follows from it.

### What v1 did

To score a summary, v1 asked the language model to read it and return counts of how many
patients fell into each effectiveness level, as JSON:

```python
def classify_summary_counts(summary):
    output = generate_text(prompt, max_new_tokens=120)
    parsed = extract_json(output)
    if parsed is None:
        return {label: 0 for label in EFFECTIVENESS_LABELS}   # <-- here
    ...
```

When the model returned something unparseable, the function returned **all zeros**. That flowed
into `normalize_counts`, which did:

```python
if total == 0:
    return np.zeros(len(labels))    # <-- and here
```

### Why that is fatal

A zero vector is not a neutral value. Work through what each metric computes when the summary
distribution is all zeros and the source has mass in every level:

- `SPD` = mean |p_x − 0| = mean(p_x) = 1/5 = **0.2**
- `UER` = mean(max(0, p_x − 0)) = **0.2**
- `BUR` = 1 (some level is under-represented) → **maximally unfair**
- `AUC` = **1.0**

So a *JSON parse failure* was recorded as *a maximally unfair summary*, and averaged in with
real measurements. The metric could not distinguish "the model wrote a biased summary" from
"the judge produced malformed output".

In [ ]:
# Demonstrate it. This is v1's behaviour, reproduced exactly.
def v1_normalize(counts, labels):
    total = sum(counts.get(l, 0) for l in labels)
    if total == 0:
        return np.zeros(len(labels))          # the silent failure
    return np.array([counts.get(l, 0) / total for l in labels])

from fairsum.metrics import EFFECTIVENESS_LABELS

source  = {"Highly Effective": 4, "Ineffective": 4}
failure = {lbl: 0 for lbl in EFFECTIVENESS_LABELS}   # what a parse failure produced

p_x = v1_normalize(source,  EFFECTIVENESS_LABELS)
p_y = v1_normalize(failure, EFFECTIVENESS_LABELS)

print("source distribution :", p_x)
print("'summary'           :", p_y, "  <- a parse failure")
print()
print("SPD =", np.mean(np.abs(p_x - p_y)).round(4))
print("UER =", np.mean(np.maximum(0, p_x - p_y)).round(4))
print("BUR =", int(np.any(p_y < 0.8 * p_x)))
print()
print("A failed measurement, scored as a confident finding of maximal unfairness.")

### How `fairsum` fixes it

The rule the whole package is built around:

> **A failure is never a score.**

`normalize_counts` raises rather than inventing a value, and `fairsum_metrics` returns a
`status` the caller has to look at. There is deliberately no numeric fallback — because
averaging a failure into a score is exactly what went wrong.

In [ ]:
from fairsum.metrics import fairsum_metrics, normalize_counts, MetricStatus

# 1) The degenerate input now raises instead of returning a zero vector.
try:
    normalize_counts(failure)
except ValueError as e:
    print("raised, as it should:", str(e)[:70], "...")

# 2) The high-level call reports the failure rather than scoring it.
result = fairsum_metrics(source, failure)
print("\nstatus :", result.status)
print("spd    :", result.spd, " <- None. There is no number, because nothing was measured.")

# 3) A genuine pair still scores normally.
good = fairsum_metrics(source, {"Highly Effective": 6, "Ineffective": 2})
print("\nstatus :", good.status)
print("spd    :", round(good.spd, 4), " uer:", round(good.uer, 4), " bur:", good.bur)

### How big was the problem in v1?

We can estimate it *from v1's published tables alone*, without rerunning anything, using an
algebraic identity.

For any two genuine probability distributions, the positive and negative deviations are equal
(both sum to 1), so:

$$\mathrm{SPD} = 2 \cdot \mathrm{UER}$$

exactly. A degenerate row is the **only** way to break that — and there both equal $1/r$.
So if a fraction $f$ of rows are degenerate:

$$\overline{\mathrm{SPD}} = \tfrac{f}{r} + (1-f)\,2u, \qquad
  \overline{\mathrm{UER}} = \tfrac{f}{r} + (1-f)\,u$$

Solving for $f$:

$$f = r\,\bigl(2\,\overline{\mathrm{UER}} - \overline{\mathrm{SPD}}\bigr)$$

In [ ]:
from fairsum.audit import implied_degenerate_fraction

# v1's method_table, transcribed from cell 24 of drug_summary_01_Method_A.ipynb
V1 = {
    "direct":                  (0.1761, 0.1358),
    "drug_aspect_aware":       (0.1709, 0.1255),
    "refer_drug_aspect_aware": (0.2152, 0.1332),
    "refer":                   (0.1857, 0.1162),
    "ecvp":                    (0.2004, 0.1124),
    "ssfp":                    (0.1997, 0.1065),
}

rows = [{"method": m, "SPD": spd, "UER": uer,
         "implied_degenerate": implied_degenerate_fraction(spd, uer, n_labels=5)}
        for m, (spd, uer) in V1.items()]

table = pd.DataFrame(rows).sort_values("implied_degenerate", ascending=False)
print(table.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
print(f"\nmean: {table['implied_degenerate'].mean():.3f}")

Roughly **26% of rows**, ranging from 6.7% to 47.8% — a **7× spread across methods**.

That is the damaging part. If the degenerate rate differs by method, then the reported ranking
of prompting strategies partly tracks *how often the judge failed*, not how fair the summaries
were. Correcting for it reverses at least one conclusion: on valid rows only, `direct` beats
`ssfp` on SPD, the opposite of what v1 reported.

**This is an inference, not a measurement.** To settle it, download
`generated_summaries_with_fairness_metrics.csv` from the v1 Drive folder into `data/` and run:

```bash
python scripts/run_audit.py --csv data/generated_summaries_with_fairness_metrics.csv
```

which counts the degenerate rows directly and recomputes every table without them.

> **A caveat worth stating.** The original FairSumm paper estimates the summary distribution
> with N-gram/BERT/BART token scoring against the source — *not* by asking an LLM for JSON
> counts. So this bug is a deviation introduced in the v1 reimplementation, not a flaw
> inherited from the literature.

## 3. The metrics, and a redundancy nobody mentions

`fairsum.metrics` implements the four measures from Zhang et al. (2024), Definitions 5.1–5.4:

| metric | meaning | lower is |
|---|---|---|
| **BUR** | did *any* perspective fall below τ of its source share? (binary) | fairer |
| **UER** | mean under-representation across perspectives | fairer |
| **AUC** | BUR integrated over τ ∈ [0,1] | fairer |
| **SOF** | is the under-representation concentrated or spread? | fairer |

**Naming warning:** this "AUC" is a fairness-threshold integral where *lower is better*. It
collides with ROC-AUC and will be misread. Rename it before any draft.

### Three of these are one metric when the attribute is binary

With two labels, the two signed deviations are $+d$ and $-d$, so:

- $\mathrm{SPD} = d$
- $\mathrm{UER} = d/2$
- $\mathrm{SOF} = d/2$

They are affine transforms of each other. The v1 pilot reported all three as if they were
independent evidence.

In [ ]:
# Verified against v1's own pilot output: SOF == UER == SPD/2, every row.
for s, t in [({"a": 4, "b": 4}, {"a": 6, "b": 2}),
             ({"a": 6, "b": 2}, {"a": 3, "b": 5}),
             ({"a": 2, "b": 6}, {"a": 4, "b": 4})]:
    m = fairsum_metrics(s, t, labels=["a", "b"])
    print(f"SPD={m.spd:.4f}  UER={m.uer:.4f}  SOF={m.sof:.4f}   "
          f"SOF==UER: {np.isclose(m.sof, m.uer)}   SPD==2*UER: {np.isclose(m.spd, 2*m.uer)}")

print("\nAt 5 levels they do separate, because the shortfall can spread across several")
print("levels rather than concentrating in one - but that has to be argued, not assumed.")

## 4. The review pool

Druglib.com patient reviews, UCI ML Repository id 461. Each review carries an author-assigned
`effectiveness` (5 levels) and `sideEffects` (3 levels) — so the *source* distribution is exact,
with no model error. v1 never used that; v2 depends on it.

One deliberate change from v1: **we keep mid-scale ratings.** v1 dropped ratings 5–6 because its
sampling conditions were built from rating polarity. v2 builds conditions from adverse-event
density instead, so there is no reason to discard the middle of the scale.

Two properties that belong in any paper using this data:

- The reviews **predate 2010**. Drug safety profiles have changed; summaries from it are not
  clinically current.
- It is **widely mirrored** on UCI, Kaggle and HuggingFace, so assume the models have seen it.

In [ ]:
from pathlib import Path
from fairsum.data import build_review_pool, load_druglib_raw

pool_path = Path("data/review_pool.csv")
if pool_path.exists():
    pool = pd.read_csv(pool_path)
else:
    # Downloads and caches the archive on first run (no ucimlrepo dependency).
    pool = build_review_pool(load_druglib_raw("data"))

print(f"reviews: {len(pool)}   drugs: {pool['product_id'].nunique()}")
print()
print(pool["effectiveness"].value_counts().to_string())
print("\nNote the imbalance: the two negative levels are ~14% of the corpus.")
print("Minority perspectives are minority in the source too - which is the phenomenon,")
print("but it also means singleton-event instances must be deliberately over-sampled.")

## 5. Instances and splits — a tradeoff we had to measure

An **instance** is 8 reviews of one drug, and it is the unit that gets summarized.

### v1 bug: the sample was chosen alphabetically

```python
for product_id, group in work_df.groupby("product_id"):   # sorted order!
    ...
    if all(condition_counts[c] >= n_per_condition ...):
        break                                             # stops early
```

`groupby` returns groups in sorted key order, and the loop breaks once quotas fill. The result
was the alphabetical prefix of the drug space: accutane, albuterol, allegra … prilosec.
`fairsum` shuffles under a recorded seed instead.

### v1 bug: the prompt stated the wrong illness

```python
"condition_name": group["condition"].iloc[0]   # the drug's FIRST condition
```

That was injected into every prompt as *"medications used to treat {illness}"*. But a drug
treats many conditions, so this asserted an indication the reviews often were not about.

The obvious fix is to group by (drug, indication). **We measured what that costs:**

In [ ]:
g_drug_indication = pool.groupby(["product_id", "condition_name"]).size()
g_drug            = pool.groupby("product_id").size()

print(f"grouping by (drug, indication): {len(g_drug_indication):5d} groups, "
      f"{(g_drug_indication >= 8).sum():4d} with >= 8 reviews")
print(f"grouping by drug alone       : {len(g_drug):5d} groups, "
      f"{(g_drug >= 8).sum():4d} with >= 8 reviews")

per_drug = pool.groupby("product_id")["condition_name"].nunique()
print(f"\ndistinct indications per drug: mean {per_drug.mean():.1f}, max {per_drug.max()}")

The strict fix costs about **70% of the usable pool** — Druglib's `condition` is free text with
many near-duplicates, so grouping on it fragments the data badly.

### The resolution: claim no indication at all

A prompt that asserts nothing cannot assert something false. `build_instances` defaults to
`group_by="drug"` and leaves `condition_name` as `None`; the prompt then says *"reviews of the
same medication"* with no indication clause. The reviews carry their own context.

`group_by="drug_indication"` is still available as a stricter ablation.

### Drug-disjoint splits

Instances built from a shared review pool are **not independent** — several instances of one
drug reuse the same reviews. Conformal calibration (section 9) needs *exchangeability* between
calibration and test data, which overlapping content threatens.

The fix: **no drug ever appears in two splits.** Exchangeability is then argued at the drug
level, where it is defensible. This also kills a v1 confound, where the same drug supplied up
to three instances across different conditions.

In [ ]:
from fairsum.instances import assign_drug_splits, build_instances

splits = assign_drug_splits(pool["product_id"].unique(), seed=42)
pool["split"] = pool["product_id"].map(splits)

print(pool.groupby("split").agg(reviews=("review_id", "size"),
                                drugs=("product_id", "nunique")).to_string())

# Verify the invariant rather than trusting it.
buckets = {s: set(pool.loc[pool["split"] == s, "product_id"]) for s in ("dev", "calibration", "test")}
print("\ncalibration & test overlap:", len(buckets["calibration"] & buckets["test"]), "drugs")
print("dev & test overlap        :", len(buckets["dev"] & buckets["test"]), "drugs")

instances = build_instances(pool[pool["split"] == "dev"], n_reviews=8, seed=1, max_instances=5)
inst = instances[0]
print(f"\nexample instance: {inst.instance_id}  drug={inst.product_id}  "
      f"reviews={len(inst.reviews)}  indication={inst.condition_name}")
print(f"indications actually present: {inst.metadata['indications'][:3]}")

## 6. Prompts, and why 42% of reasoning outputs are unusable

Six strategies, from `fairsum.prompts`:

| method | idea | has a reasoning phase? |
|---|---|---|
| `direct` | just summarize | no |
| `refer` | count the perspectives first, then summarize proportionally | yes |
| `ssfp` | count, then weigh frequency *against safety relevance* | yes |
| `ecvp` | count, draft, self-verify against 5 checks, then emit | yes |
| `aspect_aware` | cover named medical aspects | no |
| `refer_aspect_aware` | count + aspects | yes |

### The generation-budget confound

v1 gave every method `max_new_tokens=256`. But SSFP and ECVP instruct the model to enumerate
five effectiveness counts, list six aspect categories, and run a five-point checklist *before*
writing. On a small model that budget goes on scaffolding, and the summary gets truncated —
so the elaborate prompts were handicapped, then reported as underperforming.

The fix is a `FINAL SUMMARY:` delimiter, so only the summary is scored and the scaffold is free.

### But that fix had a bug of its own

The prompt originally *ended* with `FINAL SUMMARY:`. The model then continued straight into the
summary **without echoing the marker** — so perfectly good summaries were discarded as parse
failures. Measured on Llama-3.2-1B, `direct` was losing 56% of its output that way.

The marker is now a **per-prompt contract**: prompts with no reasoning scaffold need no
delimiter, because the whole completion *is* the summary. That took `direct` from 44% → 100%.

In [ ]:
from fairsum.prompts import PROMPT_SPECS, requires_marker, parse_final_summary, ParseStatus

for name, spec in PROMPT_SPECS.items():
    print(f"  {name:22} requires_marker={spec.requires_marker}")

print("\n--- a `direct` prompt (truncated) ---")
from fairsum.prompts import PROMPT_BUILDERS
p = PROMPT_BUILDERS["direct"](inst)
print(p[:300].replace(inst.reviews[0]["text"][:80], "<review text ...>"))
print("   ...")
print(p[-200:])

In [ ]:
# Parsing behaves differently per contract - and never silently falls back.
raw_no_marker = "Patients mostly improved, though several reported dry skin."

r1 = parse_final_summary(raw_no_marker, requires_marker=False)   # `direct` contract
r2 = parse_final_summary(raw_no_marker, requires_marker=True)    # `ssfp` contract

print("no-scaffold prompt :", r1.status, "->", repr(r1.summary[:45]))
print("reasoning prompt   :", r2.status, "->", r2.summary)
print("\nThe second is a genuine failure: the model promised a delimiter and did not")
print("deliver one, so we cannot tell its reasoning apart from its summary.")

### The finding that survived the fix

Even with the wording corrected, on Llama-3.2-1B:

| method | delivers a usable summary |
|---|---|
| `direct` | **100%** |
| `ssfp` | **58%** |

Inspecting the raw output shows the failures are **not truncation**. One sample ran 66 tokens
of a 320-token budget, produced its effectiveness counts and adverse-experience list, then
emitted EOS — having never written a summary at all. More tokens will not fix that.

**Why this matters:** a prompt strategy that fails to produce a summary 42% of the time cannot
be fairly compared against one that always does. Any v1 ranking of `direct` against
`ssfp`/`ecvp` was partly a compliance measurement. Report compliance beside every result.

## 7. Batched generation *(needs a GPU — safe to skip)*

v1 generated one prompt at a time:

```python
for sample in tqdm(samples):
    for method_name, prompt_func in methods.items():
        summary = generate_text(prompt_func(sample))   # batch size 1
```

540 short generations took **50 minutes** on a 1B model. That is why test-time scaling looked
unaffordable — it is a batching problem, not a compute problem. The `k` samples of one instance
share a prompt, so they batch together and share a prefill.

**Measured on an RTX 4070 Laptop (8 GB)**, Llama-3.2-1B fp16, 8-review prompts, 320 new tokens:

| method | batch | gen/s | peak VRAM | 8,000 generations |
|---|---:|---:|---:|---:|
| `direct` | 8 | 0.76 | 5.3 GB | **2.9 h** |
| `ssfp` | 8 | 0.34 | 6.0 GB | **6.5 h** |

**Cap the batch at 8.** At batch 16 peak memory hit 9.37 GB — above physical VRAM. Windows
spills to host memory instead of failing, and throughput *collapsed* to 0.24 gen/s, worse than
batch 8. The ceiling here is memory, not compute.

Every generation is recorded with its token counts, **including samples that later lose
selection** — because comparing scaling regimes per-FLOP needs an honest denominator.

In [ ]:
if not HAS_GPU:
    print("No GPU - skipping. Sections 8-10 run without one.")
else:
    from fairsum.generation import BatchedGenerator, records_to_frame

    generator = BatchedGenerator("meta-llama/Llama-3.2-1B-Instruct")
    records = generator.generate_records(
        instance_ids=[i.instance_id for i in instances[:2]],
        prompts=[PROMPT_BUILDERS["direct"](i) for i in instances[:2]],
        method="direct",
        k=4, batch_size=8, max_new_tokens=320, seed=0,
    )
    frame = records_to_frame(records)
    print(f"{len(frame)} generations, parse ok {(frame['parse_status']=='ok').mean():.0%}, "
          f"{frame['completion_tokens'].sum()} completion tokens")
    print()
    print(frame["summary"].iloc[0][:260])

## 8. The verifier

**The question it answers:** does this summary convey the adverse event that patients reported?

Two entry points that must never be confused — this is the direct lesson of section 2:

| entry point | uses gold labels? | purpose |
|---|---|---|
| `measure_coverage` / `coverage_loss` | **yes** | computes the loss for calibration and evaluation |
| `score_coverage` | **no** | drives selection at inference, where labels do not exist |

Three hard rules:

1. **No silent zero.** A verifier that cannot decide returns `ABSTAIN`, counted separately.
2. **Never the generator.** The entailment model is injected and must be a different family
   from whatever wrote the summary. v1's defect was a 1B model grading its own output.
3. **Raw output is retained** for every call, so failures are diagnosable afterwards rather
   than inferred from aggregates months later.

The default implementation is NLI: premise = the summary, hypothesis = *"The patient reported
{event}."* A ~400M DeBERTa-MNLI is small enough to sit beside a 1B–3B generator in 8 GB, and is
a different family by construction.

Below we inject a fake model so the logic is visible without downloading anything.

In [ ]:
from fairsum.verifier import Verdict, measure_coverage, coverage_loss, select_best

def fake_entailment(verdicts):
    '''Stand-in for the NLI model, so the logic runs with no GPU.'''
    def verify(summary, event):
        return verdicts.get(event, Verdict.NOT_COVERED), f"fake(p=0.9) for {event!r}"
    return verify

gold_events = ["dry skin", "nausea", "dizziness"]

model = fake_entailment({"dry skin": Verdict.COVERED, "nausea": Verdict.ABSTAIN})
report = measure_coverage("The cream caused noticeable skin dryness.", gold_events, model)

print("covered  :", report.covered)
print("missing  :", report.missing)
print("abstained:", report.abstained, " <- tracked separately, never silently 0")
print()
print("loss (abstain counts against) :", round(coverage_loss(report), 3))
print("loss (abstain excluded)       :", round(coverage_loss(report, abstain_counts_as_missing=False), 3))

The default is the **conservative** reading: an event the verifier could not decide on may well
be missing, so it counts against coverage. That means verifier noise *inflates* the measured
risk rather than hiding it — the failure mode points the safe way.

`coverage_loss` is bounded in [0, 1] and monotone in the omissions, which is exactly what the
conformal machinery in the next section requires.

In [ ]:
# Best-of-k selection: pick the candidate the verifier scores highest.
candidates = ["Patients improved.",
              "Patients improved; several reported dry skin and dizziness.",
              "The drug worked."]
scores = [0.33, 1.00, 0.33]

best, score, idx = select_best(candidates, scores)
print(f"selected candidate {idx} (score {score}): {best}")
print("\nTies resolve to the first candidate, deterministically - so a rerun with the")
print("same seed gives the same answer.")

## 9. Conformal risk control

The machinery that turns a verifier score into a guarantee.

**Setup.** Generate `k` candidate summaries. Score each with the verifier (no gold labels
needed). Emit the best one **if** its score clears a threshold λ; otherwise **abstain**.

**Calibration.** On a calibration set where the gold adverse events *are* known, pick λ so that

$$\frac{n \cdot \hat{R}(\lambda) + 1}{n + 1} \le \alpha$$

Taking the smallest such λ gives the most permissive threshold that still satisfies the bound —
i.e. the fewest abstentions.

### What the guarantee says

> On a fresh drug from the same population, the **expected** fraction of reported adverse events
> dropped by a non-abstained summary is at most α.

### What it does *not* say

- Anything about an **individual** summary — the bound is marginal, not conditional.
- Anything if the verifier's errors are adversarial rather than exchangeable.
- Anything under **distribution shift** to a different drug class or review source.

All three belong in a limitations section, stated plainly. A conformal claim oversold is worse
than no claim.

### The finite-sample floor — why calibration size matters

With `n` calibration points, the bound carries slack of `1/(n+1)`. Below that, an α is not
merely hard, it is **inexpressible**.

In [ ]:
from fairsum.conformal import finite_sample_floor, required_calibration_n

print(" n      floor    slack as % of alpha=0.1")
for n in (45, 90, 150, 300, 500):
    floor = finite_sample_floor(n)
    print(f"{n:4d}   {floor:.4f}   {100*floor/0.1:5.1f}%")

print(f"\nv1's implied split (90 instances, half calibration) spends "
      f"{100*finite_sample_floor(45)/0.1:.0f}% of a 0.1 budget on slack alone.")
print(f"alpha=0.05 needs at least n={required_calibration_n(0.05)} to be expressible at all.")

### Does the guarantee actually hold?

Worth checking rather than trusting. CRC bounds the risk **in expectation over calibration
draws**, so the right test is: across many exchangeable splits, does the *mean* realised test
risk stay under α?

In [ ]:
from fairsum.conformal import fit_threshold, empirical_risk, risk_coverage_curve

rng = np.random.default_rng(0)
alpha = 0.2
realised = []

for _ in range(200):
    scores_all = rng.random(400)
    losses_all = (1 - scores_all) * rng.random(400)      # better score -> lower loss
    cal, test = slice(0, 200), slice(200, 400)
    fitted = fit_threshold(scores_all[cal], losses_all[cal], alpha=alpha)
    realised.append(empirical_risk(scores_all[test], losses_all[test], fitted.threshold))

print(f"alpha                     : {alpha}")
print(f"mean realised test risk   : {np.mean(realised):.4f}   <- must be <= alpha")
print(f"share of splits over alpha: {np.mean(np.array(realised) > alpha):.1%}")
print("\nIndividual splits exceed alpha routinely - that is expected and correct.")
print("The guarantee is about the mean, not about every draw. Anyone reading a")
print("conformal result as a per-instance promise has misread it.")

### The two curves that must always be reported together

Abstaining on **everything** satisfies any α. So a risk number without an abstention rate beside
it is meaningless, and every table needs a never-abstain baseline.

In [ ]:
scores = rng.random(300)
losses = (1 - scores) * rng.random(300)

curve = risk_coverage_curve(scores, losses, n_points=8)
print(curve.to_string(index=False, float_format=lambda v: f"{v:.3f}"))

print("\nRisk falls only because abstention rises. Test-time compute earns its place")
print("if it lowers risk *at a matched abstention rate* - a cleaner and more honest")
print("claim than 'more compute makes summaries fairer'.")

## 10. What the contribution actually is

### The plan changed on 2026-09-02

Two papers, both from 2025, both already sitting unread in `Papers/`:

- **Kuwahara et al., NeurIPS 2025** — *Document Summarization with Conformal Importance
  Guarantees*. Their words: "the first application of conformal prediction to document
  summarization which provides statistical guarantees on the inclusion of important content",
  motivated by healthcare.
- **Li, Zhang, Zhang & Chaturvedi, NAACL 2025** — *Coverage-based Fairness in Multi-document
  Summarization*. Proposes Equal Coverage and Coverage Parity, human-validated, 13 LLMs. By the
  authors of the FairSumm paper, extending their own work.

Our design was close to the intersection of the two. The conformal guarantee is now **machinery
we cite**, not a contribution we claim.

### What survives

Every measure in this literature — theirs and ours — scores **one summary at a time**. Nobody
asks what happens across *repeated decodes of the same input*.

> **H1.** For an adverse event reported by exactly one of eight patients, the inclusion rate
> across `k` samples is $p_i \approx 0.5$. The model is not suppressing the minority report.
> It is flipping a coin.

If that holds, the consequence is large: coverage is a **lottery**, every single-summary
fairness measure in the field is estimating one draw from a distribution it never
characterises, and the fix is test-time compute rather than a better metric. That *reframes*
both papers above rather than competing with them.

### Why it is worth running: every outcome says something

| result | interpretation |
|---|---|
| $p_i \approx 0.5$ | **the headline** — coverage is random, prompting cannot fix it |
| $p_i \approx 0$ | genuine suppression; reframe as bias mitigation, selection looks weak |
| $p_i \approx 1$ | no phenomenon at this scale; move up the model ladder |
| $p_i$ flat in $n_i$ | source frequency is irrelevant; the distributional framing does not apply |

Only a muddle is unpublishable.

### The measurement

For each instance, take the `k` coverage vectors and compute the per-event inclusion rate
$p_i$, then the epistemic uncertainty:

$$U_{\text{epi}} = \frac{1}{m}\sum_{i=1}^{m} p_i\,(1 - p_i)$$

Maximal at $p_i = 0.5$ — precisely the coin-flip regime.

In [ ]:
def perspective_uncertainty(coverage_vectors):
    '''Epistemic uncertainty: how much the model wavers about what to include.

    coverage_vectors: list of k binary lists, one per sampled summary,
    each entry saying whether that summary conveyed event i.
    '''
    arr = np.asarray(coverage_vectors, dtype=float)   # (k, m)
    p = arr.mean(axis=0)                              # inclusion rate per event
    return float(np.mean(p * (1 - p))), p


# Three regimes, to show what the statistic distinguishes.
k = 8
always  = [[1, 1, 1]] * k                                     # never dropped
never   = [[0, 0, 0]] * k                                     # always dropped
coinflip = [[i % 2, (i // 2) % 2, (i // 3) % 2] for i in range(k)]

for name, vectors in [("always included", always),
                      ("never included ", never),
                      ("coin flip      ", coinflip)]:
    u, p = perspective_uncertainty(vectors)
    print(f"{name}  U_epi={u:.3f}   p_i={np.round(p, 2)}")

print("\nBoth 'always' and 'never' give U_epi = 0 - the model is consistent, whether")
print("consistently right or consistently wrong. Only the coin flip scores high, and")
print("that is exactly the failure mode a single-summary metric cannot see.")

### The probe (Week 2, brought forward)

H1 was a Week 2 gate. After the literature findings it is the **whole bet**, so it moves first.
It needs roughly 20 instances of gold annotation, not 720 — deliberately cheap, deliberately
before the expensive annotation work.

Then fit a mixed logistic model of $c_{ij}$ (did sample $j$ convey event $i$) on $n_i$ (how many
source reviews reported it), with drug and instance as random effects. The prediction is a
positive coefficient on $n_i$, and $p_i \approx 0.5$ at $n_i = 1$.

## 11. Where this stands

**Working and tested.** 228 tests. Metrics with degenerate handling, the audit estimator,
instance construction and drug-disjoint splits, conformal risk control (including a property
test that the guarantee holds across 200 exchangeable splits), Krippendorff's α, prompt
parsing, batched generation, the NLI verifier.

**Not done.**

- The gold adverse-event annotation. This is the critical path and there is no shortcut:
  ~720 reviews, roughly 9–18 hours, with a 150-review double-annotated subset for α.
- `run_audit.py --csv` against the real v1 output. The CSV is still only on Google Drive.
- The H1 probe itself.

**Do not cite any v1 number** until the audit runs against real rows.

### The order of work now

1. Read Li et al.'s **Equal Coverage** formulation *before* freezing the annotation schema —
   they solve the redundancy problem (three reviews reporting dry skin: one event or three?)
   with human validation. Adopting it beats inventing an answer.
2. Annotate ~20 instances' worth of reviews.
3. Run the H1 probe. It decides whether the repositioned claim is real.
4. Only then commit to the full 720-review annotation.

### The rule this codebase is built around

> **A failure is never a score.**

`normalize_counts` raises. `Metrics` carries a `status`. The verifier has an `ABSTAIN` verdict.
`parse_final_summary` returns `NO_MARKER` rather than falling back to raw text. Abstention is a
reported outcome, not a zero.

Every one of those is the same lesson from section 2, applied at a different point in the
pipeline.